In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import sys
import json
from collections import Counter

sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON, BOOK_TEXT, TEST_AUDIO, RECORDING_WAV, CLIPS_DIR, MANIFEST_ALL, MANIFEST_CLEAN
from book_check import check_against_book
from chunks import make_chunks
from grading import grade_chunks
from clips import decode_to_wav, write_clips, write_manifest

In [4]:
import sys
import json
from collections import Counter

sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON, BOOK_TEXT, TEST_AUDIO, RECORDING_WAV, CLIPS_DIR, MANIFEST_ALL, MANIFEST_CLEAN
from book_check import check_against_book
from chunks import make_chunks
from grading import grade_chunks
from clips import decode_to_wav, write_clips, write_manifest

In [5]:
words = json.loads(ALIGNMENT_JSON.read_text(encoding="utf-8"))
words = check_against_book(words, BOOK_TEXT.read_text(encoding="utf-8"))

confirmed = sum(word["confirmed"] for word in words)
print(f"Stage 10: {confirmed} of {len(words)} words confirmed by the book ({100 * confirmed / len(words):.1f}%)")

Stage 10: 839 of 884 words confirmed by the book (94.9%)


In [6]:
chunks = make_chunks(words)

durations = [chunk["duration"] for chunk in chunks]
print(f"Stage 11: {len(chunks)} chunks, {min(durations)}-{max(durations)} s, average {sum(durations) / len(durations):.1f} s")

Stage 11: 48 chunks, 7.16-16.56 s, average 11.6 s


In [7]:
chunks = grade_chunks(chunks, words)

print("Stage 12:", dict(Counter(chunk["tier"] for chunk in chunks)))
print()
for chunk in chunks:
    if chunk["tier"] != "clean":
        print(f"{chunk['tier']:7} {chunk['start']:7.2f} s   {'; '.join(chunk['reasons'])}")

Stage 12: {'review': 5, 'reject': 31, 'clean': 12}

review    11.00 s   not confirmed by the book: Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.; repaired word 'Qadrli' (trimmed 11.2 s)
reject    24.36 s   unsure word 'undan:' (score -9.54)
reject    35.96 s   unsure word '"Podshohim,' (score -5.0); unsure word 'nima?"' (score -9.37); unsure word '"Mol-u' (score -7.56)
reject    50.60 s   unsure word 'qolmaydi",' (score -6.18); unsure word '"Olampanoh,' (score -6.64)
reject    72.70 s   unsure word 'qoʻydiribdi:' (score -4.94); unsure word '"Kimki' (score -6.74)
reject    84.10 s   unsure word 'inʼom' (score -10.38); unsure word 'etiladi!"' (score -8.91)
reject    92.14 s   unsure word 'inʼomiga' (score -7.43)
reject   119.62 s   unsure word '–' (score -16.44)
reject   133.43 s   unsure word 'qoʻyibdi:' (score -5.69); unsure word '–' (score -17.25); unsure word '–' (score -17.27)
reject   146.27 s   unsure word '

In [8]:
decode_to_wav(TEST_AUDIO, RECORDING_WAV)
chunks = write_clips(chunks, RECORDING_WAV, CLIPS_DIR, "test")

write_manifest(chunks, MANIFEST_ALL, {"clean", "review", "reject"})
write_manifest(chunks, MANIFEST_CLEAN, {"clean"})

for tier in ["clean", "review", "reject"]:
    seconds = sum(chunk["duration"] for chunk in chunks if chunk["tier"] == tier)
    count = sum(chunk["tier"] == tier for chunk in chunks)
    print(f"{tier:7} {count:3} clips  {seconds / 60:5.2f} min")
print()
print("Clips in:", CLIPS_DIR)
print("Training manifest:", MANIFEST_CLEAN)

clean    12 clips   2.34 min
review    5 clips   0.97 min
reject   31 clips   5.98 min

Clips in: /home/intern/book_audio_alignment/data/test/chunks/wavs
Training manifest: /home/intern/book_audio_alignment/data/test/manifest_clean.jsonl


In [9]:
from IPython.display import Audio, display

for chunk in chunks:
    if chunk["tier"] != "clean":
        print(f"{chunk['tier']}  {chunk['start']:.2f} s  {'; '.join(chunk['reasons'])}")
        print("   ", chunk["text"])
        display(Audio(chunk["audio_filepath"]))

review  11.00 s  not confirmed by the book: Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.; repaired word 'Qadrli' (trimmed 11.2 s)
    Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.


reject  24.36 s  unsure word 'undan:' (score -9.54)
    Qadim zamonda bir podsho oʻtgan ekan, U taxtda doim gʻamgin oʻtirar ekan. Kunlarning birida oʻng qoʻli vaziri undan:


reject  35.96 s  unsure word '"Podshohim,' (score -5.0); unsure word 'nima?"' (score -9.37); unsure word '"Mol-u' (score -7.56)
    "Podshohim, nimagadir mudom gʻamginsiz, boisi nima?" deb soʻrabdi. "Mol-u davlatim yetarli, ammo oʻlgandan keyin bel bogʻlab boʻzlaydigan oʻgʻlim yoʻq,


reject  50.60 s  unsure word 'qolmaydi",' (score -6.18); unsure word '"Olampanoh,' (score -6.64)
    oʻgʻil koʻrsam, boshqa armonim qolmaydi", debdi podsho. "Olampanoh, sizga maslahatim bor. Chinmochin mamlakatida bir daraxt bor,


reject  72.70 s  unsure word 'qoʻydiribdi:' (score -4.94); unsure word '"Kimki' (score -6.74)
    podsho vazirdan bu maslahatni eshitib xursand boʻlibdi va shaharga jarchi qoʻydiribdi: "Kimki Chinmochin mamlakatiga borib, podsho uchun


reject  84.10 s  unsure word 'inʼom' (score -10.38); unsure word 'etiladi!"' (score -8.91)
    daraxti baxtning bargidan olib kelsa, ul bahodirga xazinaning yarmisi inʼom etiladi!"


... (26 outputs left out)

reject  514.26 s  unsure word 'kerak",' (score -8.89)
    sara qilibsan, oʻgʻillari otalarining qotilini izlab motam tutyaptilar, endi ulardan ehtiyot boʻlish kerak", debdi.


reject  536.58 s  unsure word 'inʼomlar' (score -7.28)
    yigit daraxti baxtning bargini tovoze bilan podshoga beribdi, Podsho xursand boʻlib, baxtiyorga koʻp inʼomlar hadya qilibdi va


reject  558.97 s  unsure word 'Hamma' (score -3.4)
    oradan bir yil oʻtgach, mamlakat hukmdori oʻgʻil koʻrdi degan xushxabar tarqalibdi. Hamma shod-u xurram boʻlib, murod-maqsadiga yetibdi. Podsho uchun daraxti baxt


reject  572.21 s  unsure word 'Bolalar' (score -8.22)
    izlagan Baxtiyor ham Gulnora pari bilan baxtli hayot kechiribdi. Bolalar adabiyoti orzu-niyatlaringiz kaliti.
